<a href="https://colab.research.google.com/github/ChewWJ1/ML-dating-app-behaviour/blob/main/ML_dating_app_behaviour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💘 Tying the Data Knot: Predicting Meaningful Connections
### WIA1006/WID3006 Machine Learning — Group Assignment
**Sem 2, Session 2025/2026 | FCSIT, Universiti Malaya**

---
**Project Goal:** Predict whether a dating app user will achieve a **meaningful connection**
(Mutual Match, Instant Match, Date Happened, or Relationship Formed)
based on their demographic profile and in-app behaviour.

**Task Type:** Binary Classification (`match_outcome` → Positive / Negative)

**Dataset:** `dating_app_behavior_dataset_extended1.csv` — 50,000 records × 25 features

---
## 📦 Section 1: Install & Import Libraries

In [ ]:
# Install required packages (run once in Colab)
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# Plot style
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

RANDOM_STATE = 42
print('Libraries loaded successfully')

---
## 📂 Section 2: Data Loading

In [ ]:
# ------------------------------------------------------------------
# Load from Google Drive (Colab) — uncomment these lines in Colab
# ------------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/Dataset/dating_app_behavior_dataset_extended1.csv'

# ------------------------------------------------------------------
# Load locally — comment out when using Colab Drive
# ------------------------------------------------------------------
DATA_PATH = 'dating_app_behavior_dataset_extended1.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
# Quick column overview
print('Column names and dtypes:')
for col in df_raw.columns:
    print(f'  {col:<30} dtype={df_raw[col].dtype}')

---
## 🔍 Section 3: Exploratory Data Analysis (EDA)

### 3.1 Basic Info & Statistics

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all').T

### 3.2 Missing Values & Duplicates

In [ ]:
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found')

dups = df_raw.duplicated().sum()
print(f'\nDuplicate rows: {dups}')

### 3.3 Target Variable — match_outcome

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All 10 classes
counts = df_raw['match_outcome'].value_counts()
colors = sns.color_palette('husl', len(counts))
axes[0].barh(counts.index, counts.values, color=colors)
axes[0].set_title('Distribution of All 10 Match Outcomes', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(v + 30, i, f'{v:,}', va='center', fontsize=9)

# Binary target
positive_outcomes_eda = {'Mutual Match', 'Instant Match', 'Date Happened', 'Relationship Formed'}
binary_labels = df_raw['match_outcome'].apply(
    lambda x: 'Positive' if x in positive_outcomes_eda else 'Negative'
)
binary_counts = binary_labels.value_counts()
axes[1].pie(binary_counts.values, labels=binary_counts.index,
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Binary Target Distribution\n(Positive vs Negative Outcome)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Positive (meaningful connection):    {binary_counts["Positive"]:,} ({binary_counts["Positive"]/len(df_raw)*100:.1f}%)')
print(f'Negative (no meaningful connection): {binary_counts["Negative"]:,} ({binary_counts["Negative"]/len(df_raw)*100:.1f}%)')

### 3.4 Categorical Feature Distributions

In [ ]:
cat_cols_eda = ['gender', 'sexual_orientation', 'location_type',
                'income_bracket', 'education_level', 'body_type',
                'relationship_intent', 'swipe_time_of_day', 'zodiac_sign']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols_eda):
    vc = df_raw[col].value_counts()
    axes[i].barh(vc.index, vc.values, color=sns.color_palette('husl', len(vc)))
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Count')
    axes[i].tick_params(labelsize=8)

plt.suptitle('Categorical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.5 Numerical Feature Distributions

In [ ]:
num_cols_eda = ['age', 'height_cm', 'weight_kg', 'app_usage_time_min',
                'swipe_right_ratio', 'likes_received', 'mutual_matches',
                'profile_pics_count', 'bio_length', 'message_sent_count',
                'emoji_usage_rate', 'last_active_hour']

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    axes[i].hist(df_raw[col], bins=40,
                 color=sns.color_palette('husl', 12)[i], edgecolor='white', linewidth=0.3)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_ylabel('Frequency')
    axes[i].tick_params(labelsize=8)

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.6 Numerical Features — Outlier Detection (Boxplots)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    axes[i].boxplot(df_raw[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=sns.color_palette('husl', 12)[i], alpha=0.7),
                    medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].tick_params(labelsize=8)

plt.suptitle('Outlier Detection — Boxplots', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.7 Feature vs Target — Numerical Features by Outcome

In [ ]:
# Create temporary EDA dataframe with binary outcome label
positive_outcomes_eda = {'Mutual Match', 'Instant Match', 'Date Happened', 'Relationship Formed'}
df_eda = df_raw.copy()
df_eda['outcome_label'] = df_eda['match_outcome'].apply(
    lambda x: 'Positive' if x in positive_outcomes_eda else 'Negative'
)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    pos_vals = df_eda[df_eda['outcome_label'] == 'Positive'][col]
    neg_vals = df_eda[df_eda['outcome_label'] == 'Negative'][col]
    axes[i].hist(neg_vals, bins=30, alpha=0.6, label='Negative', color='#F44336', edgecolor='none')
    axes[i].hist(pos_vals, bins=30, alpha=0.6, label='Positive', color='#4CAF50', edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

plt.suptitle('Numerical Features by Match Outcome', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.8 Feature vs Target — Categorical Features by Outcome

In [ ]:
# Stacked percentage bar charts — shows positive rate per category
cat_subset = ['gender', 'sexual_orientation', 'location_type',
              'income_bracket', 'relationship_intent', 'body_type']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_subset):
    ct = pd.crosstab(df_eda[col], df_eda['outcome_label'], normalize='index') * 100
    # Ensure both columns exist
    for c in ['Negative', 'Positive']:
        if c not in ct.columns:
            ct[c] = 0
    ct[['Negative', 'Positive']].plot(
        kind='barh', ax=axes[i], stacked=True,
        color=['#F44336', '#4CAF50'], edgecolor='white', linewidth=0.5
    )
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Percentage (%)')
    axes[i].legend(loc='lower right', fontsize=8)
    axes[i].tick_params(labelsize=8)
    axes[i].axvline(x=39.7, color='white', linestyle='--', linewidth=0.8, alpha=0.7)

plt.suptitle('Positive Match Rate by Categorical Feature', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.9 Correlation Heatmap (Numerical Features)

In [ ]:
corr_matrix = df_raw[num_cols_eda].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
plt.title('Pearson Correlation — Numerical Features', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

### 3.10 Interest Tags Analysis

In [ ]:
# Flatten all interest tags and count frequency
all_tags = [tag.strip() for tags in df_raw['interest_tags'].dropna() for tag in tags.split(',')]
tag_counts = Counter(all_tags)
tag_df = pd.DataFrame(tag_counts.most_common(30), columns=['interest', 'count'])

plt.figure(figsize=(12, 7))
colors = sns.color_palette('husl', len(tag_df))
plt.barh(tag_df['interest'], tag_df['count'], color=colors)
plt.title('Top 30 Interest Tags Among Users', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print(f'Total unique interest tags: {len(tag_counts)}')

---
## 🧹 Section 4: Data Preprocessing

### 4.1 Create Working Copy & Drop Redundant Columns

In [ ]:
df = df_raw.copy()

# Drop label/string versions of numeric columns (they add no new information)
# app_usage_time_label mirrors app_usage_time_min
# swipe_right_label mirrors swipe_right_ratio
df.drop(columns=['app_usage_time_label', 'swipe_right_label'], inplace=True)

print(f'Shape after dropping redundant columns: {df.shape}')
print('Remaining columns:', df.columns.tolist())

### 4.2 Create Binary Target Variable

In [ ]:
# Define positive outcome = any form of meaningful connection
positive_outcomes = {'Mutual Match', 'Instant Match', 'Date Happened', 'Relationship Formed'}

df['target'] = df['match_outcome'].apply(lambda x: 1 if x in positive_outcomes else 0)

print('Binary target distribution:')
vc = df['target'].value_counts()
for k, v in vc.items():
    label = 'Positive (Meaningful Connection)' if k == 1 else 'Negative (No Meaningful Connection)'
    print(f'  {k} - {label}: {v:,} ({v/len(df)*100:.1f}%)')

# Drop the original string target — no longer needed for modeling
df.drop(columns=['match_outcome'], inplace=True)

### 4.3 Encode Ordinal Feature — income_bracket (7 levels → 3 tiers)

In [ ]:
print('income_bracket unique values:', df['income_bracket'].unique())

# Consolidate 7 granular levels into 3 interpretable tiers
income_map = {
    'Very Low':     'Low',
    'Low':          'Low',
    'Lower-Middle': 'Middle',
    'Middle':       'Middle',
    'Upper-Middle': 'Middle',
    'High':         'High',
    'Very High':    'High'
}
df['income_bracket'] = df['income_bracket'].map(income_map)
print('After mapping:', df['income_bracket'].value_counts().to_dict())

# Ordinal encode: Low=0, Middle=1, High=2
df['income_enc'] = OrdinalEncoder(categories=[['Low', 'Middle', 'High']]).fit_transform(df[['income_bracket']])
df.drop(columns=['income_bracket'], inplace=True)
print('income_enc values:', sorted(df['income_enc'].unique()))

### 4.4 Encode Ordinal Feature — education_level (9 levels → 3 tiers)

In [ ]:
print('education_level unique values:', df['education_level'].unique())

# Note: CSV contains curly apostrophes (e.g. Bachelor\u2019s), so we match by keywords
def map_education(val):
    val = str(val)
    if any(k in val for k in ['No Formal', 'High School', 'Diploma']):
        return 'Low'
    elif any(k in val for k in ['Associate', 'Bachelor']):
        return 'Middle'
    elif any(k in val for k in ['Master', 'MBA', 'PhD', 'Postdoc']):
        return 'High'
    return 'Low'  # fallback

df['education_level'] = df['education_level'].apply(map_education)
print('After mapping:', df['education_level'].value_counts().to_dict())

# Ordinal encode: Low=0, Middle=1, High=2
df['education_enc'] = OrdinalEncoder(categories=[['Low', 'Middle', 'High']]).fit_transform(df[['education_level']])
df.drop(columns=['education_level'], inplace=True)
print('education_enc values:', sorted(df['education_enc'].unique()))

### 4.5 One-Hot Encode Nominal Categorical Features

In [ ]:
# These features have no natural order — use one-hot encoding
nominal_cols = [
    'gender',
    'sexual_orientation',
    'location_type',
    'swipe_time_of_day',
    'body_type',
    'relationship_intent',
    'zodiac_sign'
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=False, dtype=int)

ohe_cols = [c for c in df.columns if any(c.startswith(n + '_') for n in nominal_cols)]
print(f'Shape after one-hot encoding: {df.shape}')
print(f'One-hot encoded columns added: {len(ohe_cols)}')

### 4.6 Multi-Hot Encode Interest Tags

In [ ]:
# Each user has 3 interests (comma-separated) — create binary columns per unique tag
mlb = MultiLabelBinarizer()
interests_split = df['interest_tags'].str.split(', ')
interest_dummies = pd.DataFrame(
    mlb.fit_transform(interests_split),
    columns=['interest_' + c for c in mlb.classes_],
    index=df.index
)
df = pd.concat([df, interest_dummies], axis=1)
df.drop(columns=['interest_tags'], inplace=True)

print(f'Interest tags encoded: {len(mlb.classes_)} unique tags')
print(f'Shape after interest encoding: {df.shape}')

### 4.7 Normalize Numerical Features with StandardScaler

In [ ]:
numeric_cols = [
    'age', 'height_cm', 'weight_kg',
    'app_usage_time_min', 'swipe_right_ratio',
    'likes_received', 'mutual_matches',
    'profile_pics_count', 'bio_length',
    'message_sent_count', 'emoji_usage_rate',
    'last_active_hour'
]

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print('Numerical features normalized with StandardScaler')
print('\nPost-normalization stats (mean~0, std~1):')
df[numeric_cols].describe().loc[['mean', 'std']].round(3)

### 4.8 Final Preprocessed Dataset Overview

In [ ]:
print(f'Final dataset shape: {df.shape}')
print(f'Total features: {df.shape[1] - 1}  |  Target column: target')
print(f'\nMissing values after preprocessing: {df.isnull().sum().sum()}')
print(f'\nData types:')
print(df.dtypes.value_counts())
df.head(3)

---
## 🎯 Section 5: Feature Selection

### 5.1 Prepare Feature Matrix & Target Vector

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

print(f'Feature matrix X: {X.shape}')
print(f'Target vector  y: {y.shape}')
print(f'\nClass balance:')
print(y.value_counts().rename({0: 'Negative', 1: 'Positive'}))

### 5.2 ANOVA F-Score Feature Selection (SelectKBest)

In [ ]:
selector_f = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X, y)

f_scores = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector_f.scores_,
    'p_value': selector_f.pvalues_
}).sort_values('f_score', ascending=False).reset_index(drop=True)

print('Top 25 features by ANOVA F-Score:')
print(f_scores.head(25).to_string(index=False))

In [ ]:
top25_f = f_scores.head(25)

plt.figure(figsize=(12, 8))
colors_f = ['#4CAF50' if s > f_scores['f_score'].median() else '#90A4AE' for s in top25_f['f_score']]
plt.barh(top25_f['feature'][::-1], top25_f['f_score'][::-1], color=colors_f[::-1])
plt.xlabel('ANOVA F-Score', fontsize=11)
plt.title('Top 25 Features — ANOVA F-Score (vs match_outcome)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Mutual Information Feature Selection

In [ ]:
mi_scores = mutual_info_classif(X, y, random_state=RANDOM_STATE)

mi_df = pd.DataFrame({
    'feature': X.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False).reset_index(drop=True)

print('Top 25 features by Mutual Information:')
print(mi_df.head(25).to_string(index=False))

In [ ]:
top25_mi = mi_df.head(25)

plt.figure(figsize=(12, 8))
colors_mi = sns.color_palette('viridis', len(top25_mi))
plt.barh(top25_mi['feature'][::-1], top25_mi['mi_score'][::-1], color=colors_mi[::-1])
plt.xlabel('Mutual Information Score', fontsize=11)
plt.title('Top 25 Features — Mutual Information Score (vs match_outcome)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 Select Final Feature Set

In [ ]:
# Keep union of top-40 features from both F-score and Mutual Information rankings
top_f_features  = set(f_scores.head(40)['feature'])
top_mi_features = set(mi_df.head(40)['feature'])
selected_features = sorted(top_f_features.union(top_mi_features))

print(f'Features selected (union of top-40 F & MI): {len(selected_features)}')
print(selected_features)

X_selected = X[selected_features]
print(f'\nX_selected shape: {X_selected.shape}')

---
## 📐 Section 6: Dimensionality Reduction — PCA

### 6.1 Explained Variance Analysis

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_selected)

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_components_90 = int(np.argmax(cumvar >= 90) + 1)
n_components_95 = int(np.argmax(cumvar >= 95) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot — individual variance per component
axes[0].bar(
    range(1, min(31, len(pca_full.explained_variance_ratio_) + 1)),
    pca_full.explained_variance_ratio_[:30] * 100,
    color=sns.color_palette('husl', 30), edgecolor='white', linewidth=0.3
)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot — Individual Explained Variance', fontweight='bold')

# Cumulative explained variance
axes[1].plot(range(1, len(cumvar) + 1), cumvar, color='#4CAF50', linewidth=2)
axes[1].axhline(y=90, color='#F44336', linestyle='--', linewidth=1.5,
                label=f'90% variance ({n_components_90} components)')
axes[1].axhline(y=95, color='#FF9800', linestyle='--', linewidth=1.5,
                label=f'95% variance ({n_components_95} components)')
axes[1].fill_between(range(1, len(cumvar) + 1), cumvar, alpha=0.15, color='#4CAF50')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim([1, min(80, len(cumvar))])

plt.suptitle('PCA Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Components needed for 90% variance: {n_components_90}')
print(f'Components needed for 95% variance: {n_components_95}')
print(f'Total features before PCA:          {X_selected.shape[1]}')

### 6.2 Apply PCA (retain 95% explained variance)

In [ ]:
# We keep BOTH feature sets to compare models with and without PCA
N_COMPONENTS = n_components_95

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_selected)
X_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(N_COMPONENTS)])

print(f'X_selected shape (original features): {X_selected.shape}')
print(f'X_pca shape     (PCA-reduced):        {X_pca.shape}')
print(f'Variance retained: {pca.explained_variance_ratio_.sum()*100:.2f}%')

### 6.3 PCA Biplot — First Two Principal Components

In [ ]:
plt.figure(figsize=(9, 6))
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(X_pca), size=3000, replace=False)
colors_map = {1: '#4CAF50', 0: '#F44336'}

for label, grp_label in [(1, 'Positive'), (0, 'Negative')]:
    mask = y.values[sample_idx] == label
    plt.scatter(
        X_pca.values[sample_idx][mask, 0],
        X_pca.values[sample_idx][mask, 1],
        c=colors_map[label], label=grp_label, alpha=0.35, s=10, edgecolors='none'
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
plt.title('PCA Biplot — PC1 vs PC2 (3,000 sample points)', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## ✂️ Section 7: Train / Test Split

In [ ]:
# --- Split on ORIGINAL selected features (primary — used for most models) ---
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y   # preserves class balance in both splits
)

# --- Split on PCA-reduced features (for PCA comparison models) ---
X_train_pca, X_test_pca, _, _ = train_test_split(
    X_pca, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('=== Train / Test Split Summary ===')
print(f'  X_train:     {X_train.shape}   y_train: {y_train.shape}')
print(f'  X_test:      {X_test.shape}    y_test:  {y_test.shape}')
print(f'\n  X_train_pca: {X_train_pca.shape}')
print(f'  X_test_pca:  {X_test_pca.shape}')
print(f'\nClass balance in y_train:')
print(y_train.value_counts().rename({0: 'Negative', 1: 'Positive'}))
print(f'\nClass balance in y_test:')
print(y_test.value_counts().rename({0: 'Negative', 1: 'Positive'}))

In [ ]:
# Visualise class balance in train and test sets
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, split_y, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    vc = split_y.value_counts()
    ax.pie(vc.values, labels=['Negative', 'Positive'],
           autopct='%1.1f%%', colors=['#F44336', '#4CAF50'],
           startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
    ax.set_title(f'{title} ({len(split_y):,} samples)', fontweight='bold')

plt.suptitle('Class Distribution — Train & Test Sets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✅ Section 8: Pre-Training Checklist

Confirm all preprocessing steps completed before model training:

| Step | Detail | Status |
|---|---|---|
| Dataset loaded | 50,000 rows × 25 features | Done |
| Redundant columns dropped | `app_usage_time_label`, `swipe_right_label` | Done |
| Binary target created | `target`: 0=Negative, 1=Positive (39.7% positive) | Done |
| Ordinal encoding | `income_bracket` (3 tiers), `education_level` (3 tiers) | Done |
| One-hot encoding | gender, orientation, location, body_type, etc. | Done |
| Multi-hot encoding | `interest_tags` (49 unique tags) | Done |
| Numerical normalization | StandardScaler on 12 numeric columns | Done |
| Feature selection | ANOVA F-Score + Mutual Information (top-40 union) | Done |
| PCA | 95% variance retained | Done |
| Train/Test split | 80/20, stratified | Done |
| Missing values | None | Done |

### Objects available for model training:
| Variable | Description |
|---|---|
| `X_train`, `X_test` | Original selected features (40k/10k rows) |
| `y_train`, `y_test` | Binary target labels |
| `X_train_pca`, `X_test_pca` | PCA-reduced features |
| `RANDOM_STATE` | 42 — use in all models for reproducibility |

**Ready for model training!**